<a href="https://colab.research.google.com/github/dongtotong/sumin/blob/main/2)%EB%AA%A8%EB%8D%B8_%ED%95%99%EC%8A%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import xml.etree.ElementTree as ET
import numpy as np
import cv2
from tqdm import tqdm

BASE = '/content/drive/MyDrive/인공지능 팀플'
IMG_DIR = os.path.join(BASE, 'Dataset_Images')
XML_DIR = os.path.join(BASE, 'Dataset_XML')
NEW_MASK_DIR = os.path.join(BASE, 'Dataset_Masks_New')

os.makedirs(NEW_MASK_DIR, exist_ok=True)

In [ ]:
# 클래스별 색상(BGR), ID 매핑
COLOR_MAP = {
    'background':        (0,   0,   0),
    'sidewalk':          (255, 64,  128),
    'alley':             (192, 0,   0),
    'roadway':           (128, 0,   64),
    'bike_lane':         (192, 128, 0),
    'braille_guide_blocks': (64, 64, 0),
    'sidewalk_damaged':  (0,   128, 255),
    'caution_stairs':    (0,   0,   255),
    'caution_manhole':   (0,   255, 255),
    'caution_grating':   (128, 255, 0),
    'caution_repair_zone': (128, 0, 255),
}

CLASS_ID = {k: i for i, k in enumerate(COLOR_MAP.keys())}

for name, idx in CLASS_ID.items():
    print(f"  {idx}: {name}")

In [ ]:
import torch
import torchvision
from torchvision.io import read_image
from torchvision.ops.boxes import masks_to_boxes
from torchvision import tv_tensors
from torchvision.transforms.v2 import functional as F
from PIL import Image
import numpy as np

# 클래스 ID 매핑
CLASS_ID = {
    'background':           0,
    'sidewalk':             1,
    'alley':                2,
    'roadway':              3,
    'bike_lane':            4,
    'braille_guide_blocks': 5,
    'sidewalk_damaged':     6,
    'caution_stairs':       7,
    'caution_manhole':      8,
    'caution_grating':      9,
    'caution_repair_zone':  10,
}
NUM_CLASSES = 11

# BGR -> RGB 자동 변환 및 매핑
COLOR_TO_ID = {}
for class_name, bgr in COLOR_MAP.items():
    rgb = (bgr[2], bgr[1], bgr[0])
    COLOR_TO_ID[rgb] = CLASS_ID[class_name]

print(f"총 클래스 수 (배경 포함): {NUM_CLASSES}")
print("\nCOLOR_TO_ID 확인:")
for rgb, cid in COLOR_TO_ID.items():
    name = list(CLASS_ID.keys())[list(CLASS_ID.values()).index(cid)]
    print(f"  RGB{rgb} → {cid}: {name}")

In [ ]:
class SidewalkDataset(torch.utils.data.Dataset):
    def __init__(self, img_dir, mask_dir, transforms=None):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.transforms = transforms
        self.imgs = sorted([
            f for f in os.listdir(img_dir) if f.endswith('.jpg')
        ])

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        # 1. 파일 경로 설정
        img_name  = self.imgs[idx]
        mask_name = img_name.replace('.jpg', '.png')
        img_path  = os.path.join(self.img_dir,  img_name)
        mask_path = os.path.join(self.mask_dir, mask_name)

        # 2. 이미지 및 마스크 로드
        img  = read_image(img_path)
        mask_pil = Image.open(mask_path).convert('RGB')
        mask_np  = np.array(mask_pil)

        # 3. 색상 기반 클래스 마스크 생성
        H, W = mask_np.shape[:2]
        class_mask = np.zeros((H, W), dtype=np.int64)
        for color_rgb, class_id in COLOR_TO_ID.items():
            if class_id == 0:
                continue
            r, g, b = color_rgb
            match = (
                (mask_np[:,:,0] == r) &
                (mask_np[:,:,1] == g) &
                (mask_np[:,:,2] == b)
            )
            class_mask[match] = class_id

        # 4. 인스턴스 분리 (Connected Components)
        import cv2
        binary_masks = []
        labels_list  = []
        for class_id in range(1, NUM_CLASSES):
            class_region = (class_mask == class_id).astype(np.uint8)
            if class_region.sum() == 0:
                continue

            num_components, components = cv2.connectedComponents(class_region)
            for comp_id in range(1, num_components):
                instance = (components == comp_id).astype(np.uint8)
                if instance.sum() < 100:
                    continue
                binary_masks.append(instance)
                labels_list.append(class_id)

        # 5. 빈 마스크 예외 처리
        if len(binary_masks) == 0:
            return self.__getitem__((idx + 1) % len(self))

        # 6. Tensor 변환 및 타겟 구성
        masks  = torch.as_tensor(np.stack(binary_masks), dtype=torch.uint8)
        labels = torch.as_tensor(labels_list, dtype=torch.int64)
        boxes = masks_to_boxes(masks)

        keep = (boxes[:, 2] - boxes[:, 0] > 0) & (boxes[:, 3] - boxes[:, 1] > 0)
        boxes  = boxes[keep]
        masks  = masks[keep]
        labels = labels[keep]

        image_id = idx
        area     = (boxes[:,3] - boxes[:,1]) * (boxes[:,2] - boxes[:,0])
        iscrowd  = torch.zeros((len(labels),), dtype=torch.int64)

        img = tv_tensors.Image(img)
        target = {
            'boxes':    tv_tensors.BoundingBoxes(
                            boxes, format='XYXY', canvas_size=F.get_size(img)),
            'masks':    tv_tensors.Mask(masks),
            'labels':   labels,
            'image_id': image_id,
            'area':     area,
            'iscrowd':  iscrowd,
        }

        if self.transforms is not None:
            img, target = self.transforms(img, target)

        return img, target

In [ ]:
from torchvision.transforms import v2 as T

def get_transform(train):
    transforms = []
    if train:
        transforms.append(T.RandomHorizontalFlip(0.5))
    transforms.append(T.ToDtype(torch.float, scale=True))
    transforms.append(T.ToPureTensor())
    return T.Compose(transforms)

# 데이터셋 생성 및 샘플 확인
dataset = SidewalkDataset(IMG_DIR, NEW_MASK_DIR, get_transform(train=True))
print(f"전체 이미지 수: {len(dataset)}")

img, target = dataset[2]
print(f"\n이미지 shape: {img.shape}")
print(f"감지된 instance 수: {len(target['labels'])}")
print(f"labels: {target['labels']}")
print(f"boxes shape: {target['boxes'].shape}")
print(f"masks shape: {target['masks'].shape}")

In [ ]:
import torchvision
from torchvision.models.detection import MaskRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

# 모델 정의 및 생성
def get_model(num_classes):
    model = torchvision.models.detection.maskrcnn_resnet50_fpn(
        weights=MaskRCNN_ResNet50_FPN_Weights.DEFAULT
    )

    # box 및 mask 예측기 교체
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    hidden_layer = 256
    model.roi_heads.mask_predictor = MaskRCNNPredictor(
        in_features_mask, hidden_layer, num_classes
    )

    return model

model = get_model(NUM_CLASSES)
print(f"num_classes: {NUM_CLASSES}")

In [ ]:
from torch.utils.data import random_split

# 학습 설정
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(f"사용 장치: {device}")

dataset_train = SidewalkDataset(IMG_DIR, NEW_MASK_DIR, get_transform(train=True))
dataset_test  = SidewalkDataset(IMG_DIR, NEW_MASK_DIR, get_transform(train=False))

torch.manual_seed(42)
indices = torch.randperm(len(dataset_train)).tolist()
split   = int(len(indices) * 0.8)

dataset_train = torch.utils.data.Subset(dataset_train, indices[:split])
dataset_test  = torch.utils.data.Subset(dataset_test,  indices[split:])

print(f"학습 데이터: {len(dataset_train)}개")
print(f"테스트 데이터: {len(dataset_test)}개")

data_loader_train = torch.utils.data.DataLoader(
    dataset_train, batch_size=2, shuffle=True,
    collate_fn=lambda x: tuple(zip(*x))
)
data_loader_test = torch.utils.data.DataLoader(
    dataset_test, batch_size=1, shuffle=False,
    collate_fn=lambda x: tuple(zip(*x))
)

model = get_model(NUM_CLASSES)
model.to(device)

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

In [ ]:
def train_one_epoch(model, optimizer, data_loader, device):
    model.train()
    total_loss = 0
    for i, (images, targets) in enumerate(data_loader):
        images  = [img.to(device) for img in images]
        # image_id는 int라 .to(device) 불가, 제외하고 나머지만 device로
        targets = [{k: v.to(device) for k, v in t.items()
                    if isinstance(v, torch.Tensor)} for t in targets]

        loss_dict = model(images, targets)
        losses    = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        total_loss += losses.item()

        if i % 50 == 0:
            print(f"  step {i}/{len(data_loader)}  loss: {losses.item():.4f}")

    return total_loss / len(data_loader)

# 학습 실행
NUM_EPOCHS = 5

for epoch in range(NUM_EPOCHS):
    print(f"[Epoch {epoch+1}/{NUM_EPOCHS}]")
    avg_loss = train_one_epoch(model, optimizer, data_loader_train, device)
    lr_scheduler.step()
    print(f"  → 평균 loss: {avg_loss:.4f}\n")

save_path = os.path.join(BASE, 'mask_rcnn_sidewalk.pth')
torch.save(model.state_dict(), save_path)

In [ ]:
# mAP 계산
!pip install torchmetrics -q

import matplotlib.pyplot as plt
from torchmetrics.detection.mean_ap import MeanAveragePrecision

def evaluate_map(model, data_loader, device):
    model.eval()
    metric = MeanAveragePrecision(iou_type="segm")

    with torch.no_grad():
        for images, targets in data_loader:
            images = [img.to(device) for img in images]

            outputs = model(images)

            preds = [{
                'boxes':  o['boxes'].cpu(),
                'scores': o['scores'].cpu(),
                'labels': o['labels'].cpu(),
                'masks':  (o['masks'][:, 0] > 0.5).cpu(),
            } for o in outputs]

            tgts = [{
                'boxes':  t['boxes'].cpu(),
                'labels': t['labels'].cpu(),
                'masks':  t['masks'].cpu().bool(),
            } for t in targets]

            metric.update(preds, tgts)

    result = metric.compute()
    return result

result = evaluate_map(model, data_loader_test, device)

print(f"mAP (IoU 0.50:0.95): {result['map'].item():.4f}")
print(f"mAP (IoU 0.50):      {result['map_50'].item():.4f}")
print(f"mAP (IoU 0.75):      {result['map_75'].item():.4f}")

In [ ]:
# train loss 그래프
train_losses = [0.7355, 0.5253, 0.4697, 0.3787, 0.3582]

plt.figure(figsize=(8, 5))
plt.plot(range(1, 6), train_losses, 'b-o', label='Train Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Mask R-CNN Training Loss')
plt.legend()
plt.grid(True)
plt.show()